In [246]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("raw_data.csv")

C:\Users\darco\AppData\Local\Temp\ipykernel_22904\322543885.py:6: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("raw_data.csv")


In [224]:
df.duplicated().sum()

np.int64(541)

In [225]:
df[df.duplicated(subset = ['id','host id','host_identity_verified'])].head(5)


,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
102058,35506831,Master Bedroom with private Bathroom & Balcony,55110690425,unconfirmed,UZeyir,Queens,Maspeth,40.74056,-73.90635,United States,...,$141,1.0,1.0,11/14/2021,0.27,3.0,1.0,339.0,NaN,NaN
102059,35507383,Cozy 2 br in sunny Fort Greene apt,80193772189,verified,Sally,Brooklyn,Fort Greene,40.68701,-73.97555,United States,...,$130,3.0,38.0,11/13/2021,0.27,3.0,1.0,0.0,NaN,NaN
102060,35507935,Duplex w/ Terrace @ Box House Hotel,72991962259,verified,The Box House Hotel,Brooklyn,Greenpoint,40.73756,-73.95350,United States,...,$181,3.0,10.0,11/13/2021,0.08,3.0,30.0,32.0,NaN,NaN
102061,35508488,"Cozy, clean Greenpoint room with yard access",74975156081,verified,Dawn,Brooklyn,Greenpoint,40.72516,-73.95004,United States,...,$118,30.0,38.0,11/13/2021,0.34,5.0,2.0,324.0,NaN,NaN
102062,35509040,2BR XL Loft: Cleaning CDC guidelines implemented,85844415221,unconfirmed,Vida,Brooklyn,Greenpoint,40.72732,-73.94185,United States,...,$71,30.0,13.0,11/13/2021,0.14,4.0,28.0,336.0,NaN,NaN


In [226]:
df[df['id']== 35509040]


,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
62480,35509040,2BR XL Loft: Cleaning CDC guidelines implemented,85844415221,unconfirmed,Vida,Brooklyn,Greenpoint,40.72732,-73.94185,United States,...,$71,30.0,13.0,11/13/2021,0.14,4.0,28.0,336.0,NaN,NaN
102062,35509040,2BR XL Loft: Cleaning CDC guidelines implemented,85844415221,unconfirmed,Vida,Brooklyn,Greenpoint,40.72732,-73.94185,United States,...,$71,30.0,13.0,11/13/2021,0.14,4.0,28.0,336.0,NaN,NaN


In [227]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [228]:
drop_cols = [
    'license',
    'house_rules',
    'last review',
    'host name',
    'id',
    'host id',
    'country code',
    'NAME',
    'country',
]

df = df.drop(columns=drop_cols)

In [229]:
df.columns


Index(['host_identity_verified', 'neighbourhood group', 'neighbourhood', 'lat',
       'long', 'instant_bookable', 'cancellation_policy', 'room type',
       'Construction year', 'price', 'service fee', 'minimum nights',
       'number of reviews', 'reviews per month', 'review rate number',
       'calculated host listings count', 'availability 365'],
      dtype='object')

In [230]:
vc = df['neighbourhood'].value_counts()
cum = vc.cumsum() / vc.sum()
keep = cum[cum <= 0.85].index

df['neighbourhood'] = df['neighbourhood'].apply(lambda x: x if x in keep else 'Other')

In [231]:
rows_before = df.shape[0]
rows_before

102058

In [232]:
df = df[df['minimum nights'] >= 0]
df = df[(df['availability 365'] >= 0) & (df['availability 365'] <= 365)]

In [233]:
df[['minimum nights', 'availability 365']].describe()

,minimum nights,availability 365
count,98057.000000,98057.000000
mean,8.147567,134.418298
std,28.653840,129.815853
min,1.000000,0.000000
25%,2.000000,2.000000
50%,3.000000,90.000000
75%,5.000000,254.000000
max,5645.000000,365.000000


In [234]:
df['price'].describe()


count     97822
unique     1151
top       $206 
freq        129
Name: price, dtype: object

In [235]:
df['price'] = (
    df['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)

df['price'] = pd.to_numeric(df['price'], errors='coerce')

df['service fee'] = (
    df['service fee']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)

df['service fee'] = pd.to_numeric(df['service fee'], errors='coerce')

In [236]:
df[['price', 'service fee']].dtypes

price          float64
service fee    float64
dtype: object

In [237]:
df.isna().sum()

host_identity_verified              259
neighbourhood group                  16
neighbourhood                         0
lat                                   8
long                                  8
instant_bookable                     74
cancellation_policy                  56
room type                             0
Construction year                   166
price                               235
service fee                         262
minimum nights                        0
number of reviews                   126
reviews per month                 14918
review rate number                  278
calculated host listings count      290
availability 365                      0
dtype: int64

In [238]:
# NA reviews means 0 reviews
df['reviews per month'] = df['reviews per month'].fillna(0)

In [239]:
cat_cols = [
    'host_identity_verified',
    'neighbourhood group',
    'instant_bookable',
    'cancellation_policy'
]

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
    
num_cols = [
    'Construction year',
    'price',
    'service fee',
    'number of reviews',
    'review rate number',
    'calculated host listings count'
]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median()) 

C:\Users\darco\AppData\Local\Temp\ipykernel_22904\2626180678.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


In [240]:
df = df.dropna(subset=['lat', 'long'])

In [241]:
df.isna().sum()

host_identity_verified            0
neighbourhood group               0
neighbourhood                     0
lat                               0
long                              0
instant_bookable                  0
cancellation_policy               0
room type                         0
Construction year                 0
price                             0
service fee                       0
minimum nights                    0
number of reviews                 0
reviews per month                 0
review rate number                0
calculated host listings count    0
availability 365                  0
dtype: int64

In [247]:
df.to_csv("cleaned_data.csv", index=False)